In [39]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import dill

In [40]:
import numpy as np
from typing import List

def get_frequency(pair: tuple, tokens:list):
    frequency = 0

    for i in range(len(tokens) - 1):
        if tokens[i] == pair[0] and tokens[i+1] == pair[1]:
            frequency += 1

    return frequency

    
class BPE:
    def __init__(self, vocab_size: int):
        self.vocab_size = vocab_size

    def __unique_tokens(self, text):
        unique_tokens = list(set(text))
        unique_tokens.sort()
        return unique_tokens
    
    def fit(self, text: str):
        self.unique_tokens = self.__unique_tokens(text)

        tokens = [c for c in text]

        while len(self.unique_tokens) < self.vocab_size:

            max_frequency =  - np.inf
            max_pair = (None, None)
            for i in range(len(tokens) - 1):
                pair = tuple(tokens[i:i + 2])

                frequency = get_frequency(pair, tokens)

                if frequency > max_frequency:
                    max_frequency = frequency
                    max_pair = pair

            new_token = ''.join(max_pair)
            self.unique_tokens.append(new_token)

            new_tokens = []
            i = 0
            while i < len(tokens) - 1:
                if max_pair[0] ==  tokens[i] and  max_pair[1] ==  tokens[i+1]:
                    i += 2
                    new_tokens.append(new_token)
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            if i == len(tokens) - 1:
                new_tokens.append(tokens[i])

            tokens = new_tokens

        self.id2token = {id: token for id, token in enumerate(self.unique_tokens)}
        
        self.token2id = {token: id for id, token in enumerate(self.unique_tokens)}

    def encode(self, text: str):
        tokens2encode = []
        
        tokens = [c for c in text]
        i = 0
        while i < len(tokens):
            token = tokens[i]
            
            appropriate_tokens = [ _token for _token in self.unique_tokens if _token.startswith(token)]
            appropriate_tokens.sort(key= lambda x: len(x), reverse=True)

            for appropriate_token in appropriate_tokens:
                tkn = ''.join(tokens[i: i + len(appropriate_token)])
                if tkn == appropriate_token:
                    break

            tokens2encode.append(tkn)
            i += len(tkn)

        return [self.token2id[token] for token in tokens2encode]
            
    def decode(self, token_ids: List[int]):
        return ''.join(self.id2token[id] for id in token_ids)

    def save(self, filename):
        with open(filename, 'wb') as f:
            dill.dump(self, f)


    @classmethod
    def load(cls, filename):
        with open(filename, 'rb') as f:
            obj = dill.load(f)
                
        return obj




In [29]:
bpe = BPE(31)

In [30]:
bpe.fit('Однажды был случай в далёком Макао: макака коалу в какао макала, коала лениво какао лакала, макака макала, коала икала.')

In [31]:
bpe.unique_tokens

[' ',
 ',',
 '.',
 ':',
 'М',
 'О',
 'а',
 'б',
 'в',
 'д',
 'е',
 'ж',
 'и',
 'й',
 'к',
 'л',
 'м',
 'н',
 'о',
 'с',
 'у',
 'ч',
 'ы',
 'ё',
 'ка',
 'ла',
 'ака',
 'ко',
 ' м',
 ' мака',
 ' ко']